# Clase 04 – Data Science I

| # | Tema |
|---|------|
| 1 | Pipelines reproducibles |
| 2 | Valores ausentes – teoría (MCAR/MAR/MNAR) |
| 3 | Imputación básica (dropna, fillna, group-based) |
| 4 | Imputación avanzada (SimpleImputer, IterativeImputer, ColumnTransformer) |
| 5 | Series de tiempo – parsing y slicing de fechas |
| 6 | Zonas horarias y shift/lag |
| 7 | Resampling, rolling, EWMA y expanding |
| 8 | GroupBy y agregaciones |
| 9 | Operaciones sobre strings |
| 10 | Tipos de datos, category y downcast |
| 11 | loc / iloc y máscaras booleanas |
| 12 | Transformaciones vectorizadas (np.where, assign, np.select) |
| 13 | Matplotlib – principios de visualización |
| 14 | Matplotlib – tipos de gráficos |
| 15 | Plotly Express – gráficos interactivos |
| 16 | Storytelling con datos |

## Setup – Importaciones

In [ ]:
# numpy: operaciones numéricas vectorizadas (arrays N-dimensionales)
import numpy as np

# pandas: estructuras de datos tabulares (Series y DataFrame)
import pandas as pd

# matplotlib: librería base de visualización en Python
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker   # formateadores de ejes (ej: separador de miles)

# seaborn: visualización estadística sobre matplotlib (sintaxis más simple)
import seaborn as sns

# sklearn – módulos de preprocesamiento y modelado
from sklearn.impute import SimpleImputer        # imputación univariante
from sklearn.compose import ColumnTransformer   # aplica transformadores por columna
from sklearn.pipeline import Pipeline           # encadena pasos en orden

# IterativeImputer es experimental en versiones antiguas de sklearn
# → usamos try/except para compatibilidad
try:
    from sklearn.impute import IterativeImputer
except ImportError:
    from sklearn.experimental import enable_iterative_imputer  # activa el feature flag
    from sklearn.impute import IterativeImputer

# plotly: gráficos interactivos (HTML con hover, zoom, etc.)
try:
    import plotly.express as px
    PLOTLY = True
except ImportError:
    PLOTLY = False
    print('plotly no instalado – pip install plotly')

# Opciones globales de visualización de pandas
pd.set_option('display.max_columns', 20)           # mostrar hasta 20 columnas
pd.set_option('display.float_format', '{:.3f}'.format)  # 3 decimales en floats

# Semilla global de numpy para reproducibilidad (resultados idénticos cada ejecución)
np.random.seed(42)
print('Librerías cargadas correctamente')

---
## Unidad 1 – Pipelines Reproducibles

Un **pipeline** encadena pasos de preprocesamiento y modelado en un único
objeto que puede ajustarse, evaluarse y serializarse.

**Ventajas clave**
- **Sin data leakage**: la transformación (ej: calcular la media para imputar) se aprende
  *solo* sobre el conjunto de entrenamiento y se aplica al de test sin volver a calcularla.
- **GridSearchCV integrado**: se pueden buscar hiperparámetros de todos los pasos a la vez.
- **Serializable**: `joblib.dump(pipe, 'modelo.pkl')` guarda todo el pipeline listo para producción.

**Flujo de datos**
```
X_train → imputer.fit_transform() → scaler.fit_transform() → model.fit()
X_test  → imputer.transform()     → scaler.transform()     → model.predict()
```

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# ── Generación de datos sintéticos ──────────────────────────────────────────
# default_rng(seed) es el generador moderno de numpy (más rápido y estadísticamente mejor)
rng = np.random.default_rng(0)
n = 300

# Creamos 3 features con distribuciones normales de distintas medias y escalas
df_pipe = pd.DataFrame({
    'x1': rng.normal(10, 3, n),   # media=10, std=3
    'x2': rng.normal(5,  2, n),   # media=5,  std=2
    'x3': rng.normal(0,  1, n),   # media=0,  std=1
})
# Target real: y = 2*x1 - x2 + ruido gaussiano
df_pipe['y'] = 2*df_pipe['x1'] - df_pipe['x2'] + rng.normal(0, 1, n)

# ── Introducir NaN artificiales ──────────────────────────────────────────────
# En datos reales siempre hay valores faltantes; el pipeline debe poder manejarlos
for col in ['x1','x2','x3']:
    idx = rng.choice(n, size=15, replace=False)  # 15 filas aleatorias sin repetición
    df_pipe.loc[idx, col] = np.nan

# ── Split train/test ─────────────────────────────────────────────────────────
# test_size=0.2 → 20% de los datos para evaluación, 80% para entrenamiento
# random_state garantiza que el split sea siempre el mismo
X = df_pipe[['x1','x2','x3']]
y = df_pipe['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Construcción del Pipeline ────────────────────────────────────────────────
# Cada tupla es ('nombre_paso', objeto_transformador_o_modelo)
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # reemplaza NaN con la media de cada columna
    ('scaler',  StandardScaler()),                # normaliza: (x - media) / std → media=0, std=1
    ('model',   LinearRegression()),              # regresión lineal por mínimos cuadrados
])

# fit() encadena: imputer.fit_transform → scaler.fit_transform → model.fit
# Todo se aprende SOLO sobre X_train (sin contaminar con X_test)
pipeline.fit(X_train, y_train)

# score() en test: pipeline aplica transform (no fit) y luego predict
score = pipeline.score(X_test, y_test)
print(f'R² en test: {score:.4f}')   # R²=1 perfecto, R²=0 igual que predecir la media
print('Pasos del pipeline:', [s for s, _ in pipeline.steps])

In [ ]:
# ── Inspeccionar coeficientes del modelo ────────────────────────────────────
# named_steps permite acceder a cada paso por su nombre
modelo = pipeline.named_steps['model']
features = ['x1','x2','x3']

# El modelo aprendió coeficientes sobre datos ESCALADOS
# → x1 debería tener coef ≈ 2, x2 ≈ -1, x3 ≈ 0 (en escala original)
for feat, coef in zip(features, modelo.coef_):
    print(f'  {feat}: {coef:.4f}')

---
## Unidad 2 – Valores Ausentes: Teoría

| Tipo | Definición | Ejemplo |
|------|-----------|---------|
| **MCAR** | Missing Completely At Random | Sensor defectuoso aleatorio |
| **MAR** | Missing At Random | Mayores no reportan edad |
| **MNAR** | Missing Not At Random | Pacientes graves no responden |

### ¿Por qué importa el tipo de ausencia?
- **MCAR**: los nulos son independientes de cualquier variable. Se puede imputar o eliminar **sin introducir sesgo**.
- **MAR**: la ausencia depende de *otras* variables observadas (ej: personas mayores no declaran ingresos). Se imputa **condicionado** a esas variables.
- **MNAR**: la ausencia depende de la variable misma (ej: pacientes muy enfermos no responden). Es el caso más difícil: requiere modelos de selección o recolección adicional.

### Herramientas de diagnóstico
- `df.isnull().sum()` → conteo de nulos por columna
- `df.isnull().mean() * 100` → porcentaje de nulos
- Heatmap de nulos → detecta patrones (ej: filas con múltiples nulos a la vez → MCAR dudoso)
- **Missing indicator**: columna binaria que marca si el valor era nulo → permite al modelo aprender de la ausencia

In [ ]:
# ── Creación de dataset con nulos controlados ────────────────────────────────
rng = np.random.default_rng(1)

# Simulamos una encuesta con 100 respondentes
# p= es el vector de probabilidades para cada valor posible
df_nan = pd.DataFrame({
    # 10% de nulos en edad (MCAR: aleatoriamente no responden)
    'edad':    rng.choice([np.nan, 25, 30, 45, 50, 60], size=100,
                          p=[0.10, 0.18, 0.18, 0.18, 0.18, 0.18]),
    # 15% de nulos en ingreso (MAR: los mayores tienden a no declararlo)
    'ingreso': rng.choice([np.nan, 20000, 35000, 50000, 80000], size=100,
                          p=[0.15, 0.21, 0.21, 0.21, 0.22]),
    # 5% de nulos en score (casi completo)
    'score':   rng.choice([np.nan, 1, 2, 3, 4, 5], size=100,
                          p=[0.05, 0.19, 0.19, 0.19, 0.19, 0.19]),
    # 8% de nulos en ciudad (categoría con pocos nulos)
    'ciudad':  rng.choice([np.nan, 'BA', 'Córdoba', 'Rosario'], size=100,
                          p=[0.08, 0.31, 0.30, 0.31]),
})

# isnull() devuelve un DataFrame booleano (True=nulo)
# .mean() promedia los True (=1) → % de nulos
print('--- % de nulos por columna ---')
print((df_nan.isnull().mean() * 100).round(1).to_string())
print()
# .sum() cuenta los True → número absoluto de nulos
print('--- Conteo absoluto de nulos ---')
print(df_nan.isnull().sum())

In [ ]:
# ── Heatmap de nulos (seaborn) ───────────────────────────────────────────────
# Transponer (.T) pone columnas en el eje Y → cada fila del heatmap = una variable
# Amarillo = nulo, Violeta = dato presente (paleta 'viridis')
# Patrón visual: si los nulos se alinean en filas → probable que sea MAR o MNAR
fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(
    df_nan.isnull().T,   # True/False transpuesto
    cbar=False,          # sin barra de color (solo 2 valores)
    cmap='viridis',      # paleta perceptualmente uniforme
    ax=ax
)
ax.set_title('Mapa de valores ausentes (amarillo = NaN)')
ax.set_xlabel('Observaciones (índice de fila)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Missing Indicator ────────────────────────────────────────────────────────
# Idea: en lugar de solo imputar, guardamos si el valor era nulo
# Esto permite al modelo aprender: '¿el hecho de que ingreso sea nulo predice algo?'
# Es útil cuando la ausencia es MNAR (el nulo en sí es informativo)
df_nan['ingreso_missing'] = df_nan['ingreso'].isnull().astype(int)
# astype(int): convierte True→1 y False→0
print(df_nan[['ingreso', 'ingreso_missing']].head(10))

In [ ]:
# ── Distribución condicional: ¿cambia la edad según si ingreso es nulo? ────────
# Si sí cambia → posible MAR (la ausencia de ingreso depende de la edad)
# Si no cambia → más consistente con MCAR
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Histograma de edad cuando ingreso está presente (ingreso_missing == 0)
df_nan[df_nan['ingreso_missing'] == 0]['edad'].dropna().hist(
    ax=axes[0], bins=10, color='steelblue'
)
axes[0].set_title('Edad | ingreso presente')
axes[0].set_xlabel('Edad')

# Histograma de edad cuando ingreso es nulo (ingreso_missing == 1)
df_nan[df_nan['ingreso_missing'] == 1]['edad'].dropna().hist(
    ax=axes[1], bins=10, color='salmon'
)
axes[1].set_title('Edad | ingreso ausente')
axes[1].set_xlabel('Edad')

plt.tight_layout()
plt.show()

---
## Unidad 3 – Imputación Básica

| Método | Cuándo usarlo |
|--------|--------------|
| `dropna()` | Pocos nulos, sin sesgo si son MCAR |
| `fillna(valor)` | Rellenar con constante conocida |
| `fillna(media/mediana/moda)` | Distribución aproximadamente normal |
| `fillna(ffill/bfill)` | Series temporales |
| **Group-based** | Distribución varía por grupo |

**Regla práctica**: usar mediana cuando hay outliers (es más robusta que la media).
Usar media solo cuando la distribución es aproximadamente simétrica.

In [ ]:
# ── dropna: eliminar filas con nulos ────────────────────────────────────────
# subset=['edad','score'] → solo elimina filas donde ESTAS columnas son NaN
# (no elimina por nulos en 'ingreso' o 'ciudad')
df_clean = df_nan.dropna(subset=['edad', 'score'])

print(f'Filas originales: {len(df_nan)} | Tras dropna: {len(df_clean)}')
# ⚠ Problema de dropna: si los nulos no son MCAR, eliminamos filas con sesgo
# → el dataset resultante puede no ser representativo de la población original

In [ ]:
# ── fillna con estadísticos de cada columna ─────────────────────────────────
df_fill = df_nan.copy()   # trabajamos sobre una copia para no modificar el original

# Mediana para 'edad': más robusta ante outliers que la media
# La mediana no se ve afectada por valores extremos (ej: una persona de 120 años)
df_fill['edad'] = df_fill['edad'].fillna(df_fill['edad'].median())

# Media para 'ingreso': asumimos distribución aproximadamente simétrica
df_fill['ingreso'] = df_fill['ingreso'].fillna(df_fill['ingreso'].mean())

# Moda para 'score': es una variable ordinal discreta → el valor más frecuente tiene sentido
# .mode() devuelve una Serie (puede haber múltiples modas) → [0] toma la primera
df_fill['score'] = df_fill['score'].fillna(df_fill['score'].mode()[0])

# Moda para variable categórica
df_fill['ciudad'] = df_fill['ciudad'].fillna(df_fill['ciudad'].mode()[0])

# Verificamos que no queden nulos (excepto 'ingreso_missing' que no tiene nulos)
print('Nulos restantes:', df_fill.isnull().sum().to_dict())

In [ ]:
# ── Group-based imputation ───────────────────────────────────────────────────
# Problema de fillna(mediana_global): si el precio varía mucho por categoría,
# imputar con la mediana global introduce sesgo en cada grupo.
# Solución: imputar con la mediana DE CADA GRUPO → preserva la distribución intra-grupo

rng = np.random.default_rng(2)
df_prod = pd.DataFrame({
    'categoria': rng.choice(['A', 'B', 'C'], size=120),
    'precio':    rng.uniform(10, 500, 120),
})
# Introducimos 30 nulos aleatorios en 'precio'
idx_nan = rng.choice(120, size=30, replace=False)
df_prod.loc[idx_nan, 'precio'] = np.nan

# groupby('categoria')['precio'].transform('median')
# → Para CADA fila, devuelve la mediana de su propio grupo
# → El resultado tiene el mismo índice que df_prod → se puede asignar directo
# fillna(...) solo reemplaza donde había NaN, deja el resto intacto
df_prod['precio'] = df_prod['precio'].fillna(
    df_prod.groupby('categoria')['precio'].transform('median')
)

print('Nulos tras imputación group-based:', df_prod['precio'].isnull().sum())
print(df_prod.groupby('categoria')['precio'].describe().round(2))

---
## Unidad 4 – Imputación Avanzada (sklearn)

### SimpleImputer – 4 estrategias
| Estrategia | Tipo de columna | Nota |
|------------|----------------|------|
| `mean` | Numérica, simétrica | Sensible a outliers |
| `median` | Numérica, con outliers | Robusta |
| `most_frequent` | Numérica o categórica | Mejor para categorías |
| `constant` | Cualquiera | Requiere `fill_value` |

### IterativeImputer (MICE – Multiple Imputation by Chained Equations)
Itera sobre cada columna con NaN modelándola como **función de las demás**.
En cada iteración: `col_con_nan ~ f(todas_las_otras_cols)` → se repite hasta convergencia.
Es multivariante: captura correlaciones entre variables → mejor que imputar cada columna por separado.

### ColumnTransformer
Aplica **transformadores distintos a subconjuntos de columnas** → clave para datasets
con columnas numéricas y categóricas mezcladas.

In [ ]:
from sklearn.impute import SimpleImputer

# ── SimpleImputer ────────────────────────────────────────────────────────────
# .values convierte el DataFrame a un array numpy (requerido por sklearn)
X_num = df_nan[['edad', 'ingreso', 'score']].values

# strategy='median': para cada columna, calcula la mediana sobre los no-NaN
# y reemplaza todos los NaN de esa columna con ese valor
imp_median = SimpleImputer(strategy='median')

# fit_transform: aprende las medianas (fit) y las aplica (transform) en un paso
# Devuelve un array numpy (sin NaN)
X_imputed = imp_median.fit_transform(X_num)

df_imputed = pd.DataFrame(X_imputed, columns=['edad', 'ingreso', 'score'])
print('SimpleImputer(median) – primeras filas:')
print(df_imputed.head())

In [ ]:
# ── IterativeImputer (MICE) ──────────────────────────────────────────────────
# max_iter=10: número de rondas completas de imputación
# random_state: para reproducibilidad (el modelo interno usa componente aleatorio)
imp_iter = IterativeImputer(max_iter=10, random_state=0)
X_mice   = imp_iter.fit_transform(X_num)

# Diferencia clave vs SimpleImputer:
# - SimpleImputer imputa cada columna independientemente (sin usar las demás)
# - IterativeImputer usa las correlaciones entre columnas → imputaciones más coherentes
# Ejemplo: si 'edad' e 'ingreso' están correlacionados, MICE aprovecha eso
df_mice = pd.DataFrame(X_mice, columns=['edad', 'ingreso', 'score'])
print('IterativeImputer (MICE) – primeras filas:')
print(df_mice.head())

In [ ]:
# ── ColumnTransformer ────────────────────────────────────────────────────────
# Problema: columnas numéricas y categóricas necesitan estrategias distintas
# ColumnTransformer aplica transformadores distintos a subconjuntos de columnas
# y concatena los resultados horizontalmente

rng = np.random.default_rng(3)
n = 200
df_ct = pd.DataFrame({
    'precio':    rng.uniform(100, 5000, n),
    'cantidad':  rng.integers(1, 100, n).astype(float),
    'categoria': rng.choice(['X', 'Y', 'Z', 'W'], n),
})
# Introducimos nulos en distintas proporciones
for col, pct in [('precio', 0.10), ('cantidad', 0.08), ('categoria', 0.12)]:
    idx = rng.choice(n, int(n * pct), replace=False)
    df_ct.loc[idx, col] = np.nan

num_cols = ['precio', 'cantidad']
cat_cols = ['categoria']

# transformers: lista de tuplas ('nombre', transformador, columnas_a_aplicar)
# → 'num': imputa numéricas con mediana
# → 'cat': imputa categóricas con la moda (most_frequent)
preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'),        num_cols),
    ('cat', SimpleImputer(strategy='most_frequent'), cat_cols),
])

# El resultado es un array con columnas num primero, luego cat
arr = preprocessor.fit_transform(df_ct)
print('Shape tras ColumnTransformer:', arr.shape)   # (200, 3)
print('Nulos en resultado:', np.isnan(arr).sum())   # debe ser 0

In [ ]:
# ── Pipeline completo: ColumnTransformer → Escalado → Modelo ─────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Target sintético (en producción sería el campo a predecir)
y_ct = rng.normal(500, 100, n)

# Solo columnas numéricas para este ejemplo de pipeline
full_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # paso 1: sin NaN
    ('scaler',  StandardScaler()),                  # paso 2: normalizar
    ('ridge',   Ridge(alpha=1.0)),                  # paso 3: regresión con regularización L2
    # Ridge penaliza coeficientes grandes → reduce overfitting vs LinearRegression
])

# cross_val_score: divide en 5 folds, entrena en 4, evalúa en 1 → repite 5 veces
# → da una estimación más robusta del rendimiento real que un único split
scores = cross_val_score(full_pipe, df_ct[num_cols], y_ct, cv=5, scoring='r2')
print(f'R² cross-val: {scores.mean():.4f} ± {scores.std():.4f}')
# ± indica la variabilidad entre folds (cuanto menor, más estable el modelo)

### missingno – Visualización de patrones de ausencia

```python
import missingno as msno
msno.matrix(df)     # patrón de nulos fila a fila (igual que el heatmap de seaborn)
msno.bar(df)        # % de completitud por columna (barras verticales)
msno.heatmap(df)    # correlación de nulidad entre columnas
                    # correlación alta → las columnas tienden a ser nulas juntas
```
*(Requiere: `pip install missingno`)*

In [ ]:
try:
    import missingno as msno
    # matrix(): filas = observaciones, columnas = variables
    # Líneas blancas verticales = nulos; sparkline derecha = completitud total
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    msno.matrix(df_nan, ax=axes[0], sparkline=False)
    msno.bar(df_nan, ax=axes[1])    # altura de barra = % de valores presentes
    plt.tight_layout()
    plt.show()
except ImportError:
    print('missingno no instalado. Ejecutar: pip install missingno')
    # Alternativa manual con seaborn/pandas:
    fig, ax = plt.subplots(figsize=(6, 3))
    completitud = (1 - df_nan.isnull().mean()) * 100   # 100% = sin nulos
    completitud.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('% Completitud por columna')
    ax.set_ylabel('%')
    ax.set_ylim(0, 110)
    plt.tight_layout()
    plt.show()

---
## Unidad 5 – Series de Tiempo: Parsing y Slicing de Fechas

| Función | Propósito |
|---------|-----------|
| `pd.to_datetime(errors='coerce')` | Parsea strings a datetime; inválidos → NaT |
| `pd.date_range(start, periods, freq)` | Genera secuencia de fechas |
| `pd.period_range(start, periods, freq)` | Genera secuencia de períodos (sin hora) |
| `set_index(col_fecha)` | Usa fecha como índice → habilita slicing temporal |
| `.loc['2023-01':'2023-06']` | Slice por rango de fechas (ambos extremos inclusivos) |

**NaT** (Not a Time) es el equivalente de NaN para fechas.

**Frecuencias comunes**: `'D'`=día, `'h'`=hora, `'ME'`=fin de mes, `'W'`=semana, `'Y'`=año.

In [ ]:
# ── Parsing con errors='coerce' ─────────────────────────────────────────────
# errors='coerce': si el string no puede parsearse como fecha → NaT (no lanza error)
# errors='raise' (default): lanza ValueError ante el primer string inválido
# errors='ignore': devuelve el string original sin parsear
fechas_raw = ['2023-01-15', '2023-02-30', 'no-es-fecha', '2023-03-10', None]
#                            ↑ 30 de feb no existe         ↑ string imposible

serie_fechas = pd.to_datetime(fechas_raw, errors='coerce')
print(serie_fechas)
print(f'NaT generados: {serie_fechas.isna().sum()}')  # .isna() detecta tanto NaN como NaT

In [ ]:
# ── date_range vs period_range ───────────────────────────────────────────────
# date_range: genera Timestamps (instantes específicos con hora)
# Útil para series con hora exacta (ej: logs, sensores, transacciones)
rango_diario = pd.date_range(start='2023-01-01', periods=10, freq='D')
print('date_range diario (Timestamps):')
print(rango_diario)

# period_range: genera Periods (intervalos de tiempo sin hora exacta)
# Útil para datos mensuales/anuales donde no importa el día exacto
rango_mensual = pd.period_range(start='2023-01', periods=6, freq='M')
print('\nperiod_range mensual (Periods):')
print(rango_mensual)

In [ ]:
# ── Serie con DatetimeIndex – slicing temporal ───────────────────────────────
rng = np.random.default_rng(4)
idx = pd.date_range('2022-01-01', periods=365, freq='D')
serie = pd.Series(rng.normal(100, 15, 365), index=idx, name='ventas')

# ── Slicing con .loc en DatetimeIndex ────────────────────────────────────────
# pandas reconoce strings de fecha parciales como rangos:
# '2022-01' = todo el mes de enero 2022 (del 01 al 31, inclusive)
enero = serie.loc['2022-01']
print(f'Enero 2022: {len(enero)} días, media={enero.mean():.2f}')

# Slice por rango explícito: ambos extremos son INCLUSIVE (a diferencia de iloc)
q1 = serie.loc['2022-01-01':'2022-03-31']
print(f'Q1 2022: {len(q1)} días, media={q1.mean():.2f}')

In [ ]:
# ── Extracción de componentes temporales con .dt ─────────────────────────────
# El accesor .dt expone los atributos de datetime (análogo a .str para strings)
df_ts = pd.DataFrame({'fecha': idx, 'ventas': serie.values})

df_ts['año']        = df_ts['fecha'].dt.year          # 2022
df_ts['mes']        = df_ts['fecha'].dt.month         # 1-12
df_ts['dia_semana'] = df_ts['fecha'].dt.dayofweek     # 0=Lunes ... 6=Domingo
df_ts['trimestre']  = df_ts['fecha'].dt.quarter       # 1-4
# Otros útiles: .day, .hour, .week, .day_name(), .is_month_end

print(df_ts.head())

---
## Unidad 6 – Zonas Horarias y Shift/Lag

| Método | Cuándo usarlo |
|--------|--------------|
| `tz_localize('UTC')` | Serie **naive** (sin zona) → asignarle una |
| `tz_convert('America/Argentina/Buenos_Aires')` | Serie **aware** → cambiar zona |
| `serie.shift(1)` | **Lag-1**: desplaza 1 período hacia adelante (introduce NaN al inicio) |
| `serie.shift(-1)` | **Lead-1**: desplaza hacia atrás (introduce NaN al final) |

**Serie naive**: no tiene información de zona horaria.
**Serie aware**: tiene zona horaria explícita.

**¿Por qué importa?** Al combinar datos de distintas fuentes (logs en UTC, sensores locales)
es crucial convertir todo a una zona común antes de hacer joins o cálculos.

In [ ]:
# ── tz_localize → tz_convert ─────────────────────────────────────────────────
# Creamos una serie con zona horaria UTC explícita (tz='UTC' en date_range)
idx_utc = pd.date_range('2023-06-01', periods=6, freq='h', tz='UTC')
serie_utc = pd.Series([100, 110, 105, 120, 115, 130], index=idx_utc, name='temperatura')

# tz_convert: convierte de UTC a la zona local de Argentina (UTC-3)
# Buenos Aires no tiene horario de verano → siempre UTC-3
# Otras ciudades con DST usarían 'America/New_York', 'Europe/Berlin', etc.
serie_arg = serie_utc.tz_convert('America/Argentina/Buenos_Aires')

print('UTC:')
print(serie_utc)
print('\nArgentina (UTC-3): las mismas horas pero 3 horas menos')
print(serie_arg)
# Notar: los valores son idénticos, solo cambia la etiqueta del índice

In [ ]:
# ── shift() para crear features de lag ───────────────────────────────────────
# Lag-k: el valor de k períodos atrás → feature muy común en series temporales
# Uso típico: predecir las ventas de este mes usando las ventas del mes pasado

rng = np.random.default_rng(5)
idx_m = pd.date_range('2022-01', periods=24, freq='ME')  # 24 meses
ventas_m = pd.Series(rng.integers(1000, 5000, 24), index=idx_m, name='ventas')

df_lag = pd.DataFrame({'ventas': ventas_m})

# shift(1): desplaza 1 posición hacia abajo → el valor de enero aparece en febrero
# → la primera fila queda NaN (no hay dato anterior)
df_lag['lag_1'] = df_lag['ventas'].shift(1)
df_lag['lag_2'] = df_lag['ventas'].shift(2)  # dos meses atrás

# Variación absoluta respecto al mes anterior
df_lag['variacion'] = df_lag['ventas'] - df_lag['lag_1']

# pct_change() es equivalente a (x - x.shift(1)) / x.shift(1) * 100
# pero más legible y manejando correctamente el signo
df_lag['pct_var'] = df_lag['ventas'].pct_change() * 100

print(df_lag.head(8).round(2))

In [ ]:
# ── Visualización: serie original vs lag-1 ───────────────────────────────────
# El lag-1 es idéntico a la serie original pero desplazado un período a la derecha
# → se puede ver cómo el valor actual 'sigue' al valor anterior
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_lag.index, df_lag['ventas'], label='Ventas', marker='o', ms=4)
ax.plot(df_lag.index, df_lag['lag_1'],  label='Lag-1',  linestyle='--', alpha=0.7)
ax.set_title('Ventas mensuales y Lag-1')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## Unidad 7 – Resampling, Rolling, EWMA y Expanding

| Operación | Descripción | Resultado |
|-----------|-------------|-----------|
| `resample('D').sum()` | Agrupación temporal (downsample) | Menos filas |
| `resample('D').ffill()` | Rellenado temporal (upsample) | Más filas |
| `rolling(7).mean()` | Ventana deslizante de tamaño fijo | Mismas filas, NaN al inicio |
| `ewm(span=7).mean()` | Ponderación exponencial decreciente | Mismas filas |
| `expanding().mean()` | Ventana que crece desde el inicio | Mismas filas |

**Rolling vs EWMA**: Rolling pondera igual todos los valores de la ventana.
EWMA da más peso a los valores recientes → reacciona más rápido a cambios.

In [ ]:
# ── Generación de serie horaria ──────────────────────────────────────────────
rng = np.random.default_rng(6)
# 24h * 90 días = 2160 observaciones horarias
idx_h = pd.date_range('2023-01-01', periods=24*90, freq='h')
# Poisson: distribución de conteos (ej: ventas por hora)
# media=50 pedidos por hora, números enteros no negativos
ventas_h = pd.Series(rng.poisson(50, len(idx_h)), index=idx_h, name='ventas')

# ── Downsample: reducir granularidad ─────────────────────────────────────────
# resample actúa como un groupby temporal
ventas_d = ventas_h.resample('D').sum()    # suma todas las horas del día → venta diaria
ventas_w = ventas_h.resample('W').mean()   # promedio de cada semana
ventas_m = ventas_h.resample('ME').sum()   # suma mensual (ME = Month End)

print(f'Horario: {len(ventas_h)} obs | Diario: {len(ventas_d)} | Semanal: {len(ventas_w)} | Mensual: {len(ventas_m)}')
print('\nPrimeras filas diarias:')
print(ventas_d.head())

In [ ]:
# ── Rolling / EWMA / Expanding ───────────────────────────────────────────────

# Rolling(7): promedio de los últimos 7 días → suaviza ruido de corto plazo
# Las primeras 6 filas son NaN (no hay 7 valores aún)
rolling_7  = ventas_d.rolling(window=7).mean()

# ewm(span=7): Media Móvil Exponencial con span=7
# El peso del valor de hace k períodos es proporcional a (1-α)^k
# donde α = 2/(span+1) = 2/8 = 0.25
# → no tiene NaN al inicio (puede calcular con 1 solo valor)
ewma_7     = ventas_d.ewm(span=7).mean()

# expanding(): ventana que crece desde el inicio hasta la fila actual
# expanding().mean() en la fila i = promedio de las filas 0..i
# → equivalente a una media acumulada creciente
expanding_ = ventas_d.expanding().mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ventas_d.index,   ventas_d,    alpha=0.3, color='gray',   label='Diario (raw)')
ax.plot(rolling_7.index,  rolling_7,   color='steelblue',          label='Rolling 7d')
ax.plot(ewma_7.index,     ewma_7,      color='orange',             label='EWMA span=7')
ax.plot(expanding_.index, expanding_,  color='green', linestyle=':', label='Expanding (acumulado)')
ax.set_title('Suavizamiento de serie temporal')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Upsample: aumentar granularidad ─────────────────────────────────────────
# Caso de uso: tenemos datos mensuales pero necesitamos índice diario
# para hacer un join con otra tabla diaria
ventas_m_up = ventas_m.resample('D').ffill()
# ffill (forward fill): copia el último valor conocido hacia adelante
# → todos los días del mes tienen el mismo valor (el del inicio del mes)

print(f'Mensual: {len(ventas_m)} obs → Upsample diario: {len(ventas_m_up)} obs')
print(ventas_m_up.head())

---
## Unidad 8 – GroupBy y Agregaciones

El patrón **Split-Apply-Combine**:
1. **Split**: dividir el DataFrame por los valores de una o más columnas
2. **Apply**: aplicar una función a cada grupo
3. **Combine**: combinar los resultados en un nuevo DataFrame

In [ ]:
# ── Dataset de calidad del aire ──────────────────────────────────────────────
# AQI = Air Quality Index: indicador estándar de calidad del aire
# PM2.5/PM10 = partículas en suspensión (por diámetro en micrones)
# NO2 = dióxido de nitrógeno (principalmente de vehículos)
rng = np.random.default_rng(7)
n = 500
df_air = pd.DataFrame({
    'ciudad': rng.choice(['Buenos Aires', 'Córdoba', 'Rosario', 'Mendoza'], n),
    'mes':    rng.integers(1, 13, n),
    'pm25':   rng.uniform(5, 150, n).round(1),
    'pm10':   rng.uniform(10, 250, n).round(1),
    'no2':    rng.uniform(5, 80, n).round(1),
    'aqi':    rng.integers(20, 300, n),
})
print(df_air.head())

In [ ]:
# ── GroupBy simple ────────────────────────────────────────────────────────────
# groupby('ciudad'): agrupa las filas por valor de 'ciudad'
# ['aqi']: selecciona solo la columna aqi para la agregación
# .mean(): calcula la media de aqi en cada grupo
# .sort_values(ascending=False): ordena de mayor a menor (ciudad más contaminada primero)
print('Media AQI por ciudad:')
print(df_air.groupby('ciudad')['aqi'].mean().sort_values(ascending=False).round(2))

In [ ]:
# ── Múltiples agregaciones con .agg() ────────────────────────────────────────
# .agg() permite aplicar múltiples funciones a múltiples columnas en un solo paso
# Sintaxis: nombre_columna_resultado = ('columna_original', 'función')
resumen = df_air.groupby('ciudad').agg(
    aqi_media   = ('aqi',  'mean'),   # media del AQI
    aqi_max     = ('aqi',  'max'),    # peor día registrado
    pm25_media  = ('pm25', 'mean'),   # media de partículas finas
    n_registros = ('aqi',  'count'),  # cuántas mediciones tiene cada ciudad
).round(2)
print(resumen)

In [ ]:
# ── Visualización GroupBy ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot: muestra la distribución completa (mediana, cuartiles, outliers) por grupo
# Útil para comparar si la distribución del AQI difiere entre ciudades
df_air.boxplot(column='aqi', by='ciudad', ax=axes[0])
axes[0].set_title('Distribución AQI por ciudad')
axes[0].set_xlabel('')

# pivot_table: crea tabla con index=mes, columns=ciudad, values=promedio de aqi
# Equivalente a un groupby(['mes','ciudad']).mean() pero con formato matricial
pivot = df_air.pivot_table(values='aqi', index='mes', columns='ciudad', aggfunc='mean')
pivot.plot(ax=axes[1], marker='o')
axes[1].set_title('AQI mensual promedio por ciudad')
axes[1].set_xlabel('Mes')

plt.suptitle('')   # elimina el título automático de pandas
plt.tight_layout()
plt.show()

---
## Unidad 9 – Operaciones sobre Strings

pandas expone el accesor `.str` para operaciones vectorizadas sobre columnas de texto.
**Vectorizado** = se aplica a toda la columna sin un for-loop explícito en Python → mucho más rápido.

| Método | Descripción |
|--------|-------------|
| `.str.lower()` / `.str.upper()` | Normalización de mayúsculas |
| `.str.strip()` | Eliminar espacios al inicio y al final |
| `.str.contains(pat, regex=True)` | Filtro con regex (devuelve bool) |
| `.str.replace(pat, repl)` | Reemplazo con regex |
| `.str.split(sep).str[n]` | Dividir por separador y acceder al elemento n |
| `.str.extract(r'pattern')` | Extraer grupos de captura de regex |

In [ ]:
df_str = pd.DataFrame({
    'nombre': ['  Juan Pérez ', 'Ana García', 'CARLOS LOPEZ', 'maria gomez', ' Pedro  '],
    'email':  ['juan@gmail.com', 'ana@yahoo.com', 'carlos@empresa.ar', 'MARIA@GMAIL.COM', 'pedro@out.com'],
    'codigo': ['PRD-001-A', 'PRD-002-B', 'SRV-003-A', 'SRV-004-C', 'PRD-005-B'],
})

# ── Normalización de nombre ──────────────────────────────────────────────────
# strip(): elimina espacios y \t al inicio y al final
# title(): pone en mayúscula la primera letra de cada palabra
df_str['nombre_clean'] = df_str['nombre'].str.strip().str.title()

# lower(): convierte a minúsculas → estándar para comparar emails
df_str['email_clean'] = df_str['email'].str.lower()

# ── Extraer dominio del email ─────────────────────────────────────────────────
# split('@'): parte el string por '@' → devuelve una Series de listas
# .str[1]: accede al segundo elemento de cada lista (el dominio)
df_str['dominio'] = df_str['email_clean'].str.split('@').str[1]

# ── Extraer partes del código con regex ──────────────────────────────────────
# extract(r'patrón'): busca grupos de captura () y los pone en columnas separadas
# ([A-Z]+)  → grupo 1: letras mayúsculas (tipo: 'PRD' o 'SRV')
# -(\d+)-   → guion, grupo 2: dígitos (número: '001','002',...)
# ([A-Z])   → grupo 3: una letra (categoría: 'A','B','C')
df_str[['tipo', 'num', 'cat']] = df_str['codigo'].str.extract(r'([A-Z]+)-(\d+)-([A-Z])')

print(df_str[['nombre_clean', 'dominio', 'tipo', 'num', 'cat']])

In [ ]:
# ── Filtros con str.contains / str.startswith ────────────────────────────────
# contains(): busca si el string contiene el patrón (regex por defecto)
# Devuelve Serie booleana → se usa como máscara
gmail_users = df_str[df_str['email_clean'].str.contains('gmail')]
print('Usuarios con gmail:', gmail_users['nombre_clean'].tolist())

# startswith(): más eficiente que contains('^PRD') para prefijos fijos
productos = df_str[df_str['codigo'].str.startswith('PRD')]
print('Productos:', productos['codigo'].tolist())

In [ ]:
# ── Frecuencia de valores en una columna de strings ─────────────────────────
# value_counts(): cuenta cuántas veces aparece cada valor único
# Devuelve Serie ordenada de mayor a menor por defecto
print(df_str['dominio'].value_counts())

---
## Unidad 10 – Tipos de Datos, Category y Downcast

Pandas usa por defecto `int64` y `float64` → máxima precisión pero mayor memoria.
En datasets grandes, optimizar los tipos puede reducir la memoria **hasta 8x**.

| Tipo | Bytes | Rango |
|------|-------|-------|
| `int64` | 8 | ±9.2×10¹⁸ |
| `int32` | 4 | ±2.1×10⁹ |
| `int16` | 2 | ±32,767 |
| `int8` | 1 | ±127 |
| `float64` | 8 | ~15 decimales significativos |
| `float32` | 4 | ~7 decimales significativos |
| `category` | variable | depende de la cardinalidad |

**Cardinalidad**: cantidad de valores únicos. `category` es eficiente cuando cardinalidad << n_filas.

In [ ]:
rng = np.random.default_rng(8)
n = 100_000   # 100.000 filas para ver diferencia de memoria

df_types = pd.DataFrame({
    'id':      rng.integers(0, 1_000_000, n),  # int64 por defecto
    'edad':    rng.integers(18, 80, n),          # rango pequeño (18-80)
    'puntaje': rng.uniform(0, 100, n),           # float64 por defecto
    'ciudad':  rng.choice(['Buenos Aires', 'Córdoba', 'Rosario', 'Mendoza', 'Tucumán'], n),
    # ciudad tiene solo 5 valores únicos sobre 100.000 filas → alta repetición
    'activo':  rng.integers(0, 2, n),            # solo 0 o 1
})

print('--- Tipos y memoria originales ---')
print(df_types.dtypes)
# deep=True: cuenta también la memoria de strings (importante para columnas de texto)
print(f'Memoria total: {df_types.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
# ── Optimización de tipos de datos ──────────────────────────────────────────
df_opt = df_types.copy()

# pd.to_numeric(downcast='unsigned'): elige el tipo entero más pequeño que contiene el rango
# 'id' (0 a 999.999) → cabe en uint32 (4 bytes) en lugar de int64 (8 bytes)
df_opt['id']   = pd.to_numeric(df_opt['id'],   downcast='unsigned')
# 'edad' (18-80) → cabe en uint8 (1 byte, rango 0-255)
df_opt['edad'] = pd.to_numeric(df_opt['edad'], downcast='unsigned')

# float32 = 4 bytes vs float64 = 8 bytes → mitad de memoria
# Pérdida de precisión: de 15 a ~7 decimales significativos
# Para puntajes (0-100) con 1-2 decimales, float32 es más que suficiente
df_opt['puntaje'] = df_opt['puntaje'].astype('float32')

# category: internamente pandas guarda un array de enteros pequeños (códigos)
# y una lista de categorías → eficiente cuando hay pocos valores únicos
# 5 ciudades en 100.000 filas: en lugar de 100.000 strings, guarda 5 strings + 100.000 ints
df_opt['ciudad'] = df_opt['ciudad'].astype('category')

# bool: 1 byte (vs 8 bytes de int64)
df_opt['activo'] = df_opt['activo'].astype('bool')

print('--- Tipos optimizados ---')
print(df_opt.dtypes)
print(f'Memoria optimizada: {df_opt.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
# ── to_numpy() ───────────────────────────────────────────────────────────────
# Convierte DataFrame a array numpy → necesario para muchos algoritmos de sklearn
# dtype del array depende de los tipos de las columnas (busca el común más amplio)
arr_numpy = df_opt[['id', 'edad', 'puntaje']].to_numpy()
print('Tipo numpy:', arr_numpy.dtype)   # float32 (el más amplio de los 3)
print('Shape:', arr_numpy.shape)

# ── Propiedades del tipo category ────────────────────────────────────────────
# .cat.categories: lista de valores únicos ordenados (índice del diccionario)
print('\nCategorías ciudad:', df_opt['ciudad'].cat.categories.tolist())
# .cat.codes: el 'código' entero de cada fila → es lo que realmente se almacena
print('Códigos internos (primeros 5):', df_opt['ciudad'].cat.codes.head().tolist())

---
## Unidad 11 – loc, iloc y Máscaras Booleanas

| Selector | Referencia | Extremo final | Ejemplo |
|----------|-----------|---------------|---------|
| `loc` | etiquetas del índice | **inclusivo** | `df.loc[0:5]` incluye la fila 5 |
| `iloc` | posición numérica (0-based) | **exclusivo** | `df.iloc[0:5]` no incluye la posición 5 |
| Máscara booleana | condición lógica | — | `df[df['col'] > 10]` |

**Operadores booleanos en pandas** (NO usar `and`, `or`, `not`):  
- `&` → AND
- `|` → OR  
- `~` → NOT  
**Siempre usar paréntesis**: `(cond1) & (cond2)` por precedencia de operadores.

In [ ]:
rng = np.random.default_rng(9)
df_loc = pd.DataFrame({
    'nombre': [f'Cliente_{i}' for i in range(20)],
    'edad':   rng.integers(18, 70, 20),
    'ventas': rng.uniform(100, 10000, 20).round(2),
    'region': rng.choice(['Norte', 'Sur', 'Centro'], 20),
})

# ── loc: selección por etiquetas ─────────────────────────────────────────────
# En este caso el índice es 0,1,2,...19 (enteros, igual que iloc)
# → loc[3:6] incluye filas con etiquetas 3, 4, 5, 6 (CUATRO filas)
# iloc[3:6] devolvería filas en posiciones 3, 4, 5 (TRES filas)
print('loc filas 3-6 (etiquetas, inclusive), columnas nombre y ventas:')
print(df_loc.loc[3:6, ['nombre', 'ventas']])

In [ ]:
# ── iloc: selección por posición ─────────────────────────────────────────────
# El slicing de iloc funciona igual que el slicing de Python (listas, arrays):
# [:3] = posiciones 0, 1, 2 (excluye la 3)
# [:2] = columnas en posición 0 y 1
print('iloc primeras 3 filas, primeras 2 columnas (exclusivo):')
print(df_loc.iloc[:3, :2])

In [ ]:
# ── Máscaras booleanas ────────────────────────────────────────────────────────

# AND (&): AMBAS condiciones deben ser True
# Paréntesis OBLIGATORIOS: & tiene mayor precedencia que ==, >, etc.
mask = (df_loc['region'] == 'Norte') & (df_loc['ventas'] > 5000)
print('Clientes de Norte con ventas > 5000:')
print(df_loc[mask])

# OR (|): AL MENOS UNA condición debe ser True
mask2 = (df_loc['edad'] < 30) | (df_loc['ventas'] > 8000)
print(f'\nJóvenes (<30 años) O con ventas altas (>8000): {mask2.sum()} registros')

# NOT (~): invierte la máscara
no_centro = df_loc[~(df_loc['region'] == 'Centro')]
print(f'Fuera de Centro: {len(no_centro)} registros')

In [ ]:
# ── Asignación condicional con loc ───────────────────────────────────────────
# Patrón: df.loc[máscara, 'columna'] = valor
# Más eficiente y legible que usar apply() o np.where para este caso
df_loc['segmento'] = 'Regular'                                      # valor por defecto
df_loc.loc[df_loc['ventas'] > 7000, 'segmento'] = 'Premium'        # sobreescribe los Premium
df_loc.loc[df_loc['ventas'] < 1000, 'segmento'] = 'Bajo'           # sobreescribe los Bajos
# ⚠ El orden importa: si Premium y Bajo se solapan, gana el último que se ejecuta
print(df_loc['segmento'].value_counts())

---
## Unidad 12 – Transformaciones Vectorizadas

**¿Por qué evitar `.apply()` con funciones Python?**
- `.apply(func)` es un for-loop disfrazado → lento en columnas grandes.
- `np.where`, `np.select` y operaciones de pandas son **operaciones C** → 10-100x más rápidas.

| Herramienta | Uso | Condiciones |
|-------------|-----|------------|
| `np.where(cond, T, F)` | Condición binaria | 1 |
| `np.select(conds, vals, default)` | Múltiples condiciones | N |
| `df.assign(col=expr)` | Agregar columna, encadenable | — |
| `df.eval('expr')` | Expresión aritmética en string | — |

In [ ]:
rng = np.random.default_rng(10)
n = 1000
df_v = pd.DataFrame({
    'quantity': rng.integers(1, 30, n),
    'price':    rng.uniform(10, 500, n).round(2),
})

# ── np.where: condición BINARIA ───────────────────────────────────────────────
# np.where(condición, valor_si_True, valor_si_False)
# Equivalente vectorizado de: if quantity > 10: 0.10 else 0.0
# Aplica la condición a todo el array de una vez (en C)
df_v['descuento']    = np.where(df_v['quantity'] > 10, 0.10, 0.0)
df_v['precio_final'] = df_v['price'] * (1 - df_v['descuento'])
print('np.where – condición binaria:')
print(df_v.head())

In [ ]:
# ── np.select: MÚLTIPLES condiciones ─────────────────────────────────────────
# np.select(lista_de_condiciones, lista_de_valores, default='si_ninguna_aplica')
# Las condiciones se evalúan en ORDEN → la primera que sea True gana
# (igual que un if/elif/elif/else)
condiciones = [
    df_v['quantity'] <= 5,    # si qty ≤ 5 → Unidad
    df_v['quantity'] <= 12,   # si qty ≤ 12 (y > 5) → Pack
    df_v['quantity'] <= 20,   # si qty ≤ 20 (y > 12) → Caja
]
opciones = ['Unidad', 'Pack', 'Caja']
# default: el valor cuando NINGUNA condición es True (qty > 20)
df_v['tipo_venta'] = np.select(condiciones, opciones, default='Mayorista')
print('\nnp.select – múltiples condiciones:')
print(df_v['tipo_venta'].value_counts())

In [ ]:
# ── assign: forma funcional y encadenable ────────────────────────────────────
# .assign() devuelve una COPIA con la nueva columna → permite encadenar con ()
# lambda d: usa el DataFrame tal como está en ese punto de la cadena
# → podemos referenciar columnas creadas en pasos anteriores de la cadena
df_v2 = (df_v
    .assign(revenue         = lambda d: d['price'] * d['quantity'])
    .assign(revenue_con_dto = lambda d: d['precio_final'] * d['quantity'])
    # ahorro = diferencia entre revenue sin y con descuento
    .assign(ahorro          = lambda d: d['revenue'] - d['revenue_con_dto'])
)
print('Totales por columna:')
print(df_v2[['revenue', 'revenue_con_dto', 'ahorro']].sum().round(2))

In [ ]:
# ── eval: expresiones aritméticas en string ──────────────────────────────────
# Ventaja: pandas compila la expresión en C usando numexpr si está instalado
# → especialmente eficiente en DataFrames muy grandes (millones de filas)
# Los nombres de columna se referencian directamente sin 'df['...']'
df_v['margen'] = df_v.eval('(price - 8) / price * 100')
# Equivalente en pandas normal: (df_v['price'] - 8) / df_v['price'] * 100
print('Margen medio:', df_v['margen'].mean().round(2), '%')

---
## Desafío – Análisis de Bitcoin

Aplicamos las herramientas de series temporales a datos de precios de criptomonedas.
**Media Móvil (MA)**: suaviza el precio y se usa como señal de tendencia.
**Retorno diario**: `(P_t - P_{t-1}) / P_{t-1}` → distribución aproximadamente normal (log-retorno es más preciso).

In [ ]:
# ── Carga de datos Bitcoin ─────────────────────────────────────────────────
try:
    # parse_dates=['Date']: convierte la columna 'Date' a datetime al cargar
    # index_col='Date': usa la fecha como índice → habilita slicing temporal
    df_btc = pd.read_csv('bitcoin_data.csv', parse_dates=['Date'], index_col='Date')
    print('Dataset Bitcoin cargado.')
except FileNotFoundError:
    # Simulamos el precio de Bitcoin con un modelo de caminata aleatoria geométrica
    # (es el modelo más simple de precios financieros: retornos normales independientes)
    rng = np.random.default_rng(11)
    idx_btc = pd.date_range('2020-01-01', periods=365*4, freq='D')
    # cumprod de (1 + retorno_diario) → precio acumulado desde precio inicial 10.000
    precio_btc = 10000 * np.cumprod(1 + rng.normal(0.001, 0.04, len(idx_btc)))
    df_btc = pd.DataFrame({
        'Close':  precio_btc,
        'Volume': rng.uniform(1e9, 5e10, len(idx_btc))
    }, index=idx_btc)
    print('Dataset sintético Bitcoin creado.')

print(df_btc.describe().round(2))

In [ ]:
# ── Media Móvil 30 días + Retorno diario ─────────────────────────────────────
# MA-30: promedio de los últimos 30 días de precio
# → cuando el precio cruza la MA-30 hacia arriba → señal alcista (en análisis técnico)
df_btc['MA_30']   = df_btc['Close'].rolling(30).mean()

# pct_change() = (Close_hoy - Close_ayer) / Close_ayer
# * 100 para expresarlo en porcentaje
df_btc['retorno'] = df_btc['Close'].pct_change() * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
# sharex=True: ambos subplots comparten el eje X → el zoom aplica a los dos

axes[0].plot(df_btc.index, df_btc['Close'],  alpha=0.6, label='Precio')
axes[0].plot(df_btc.index, df_btc['MA_30'],  label='MA 30d', linewidth=2)
axes[0].set_title('Bitcoin – Precio y Media Móvil 30 días')
axes[0].legend()

# Retorno coloreado: verde si positivo, rojo si negativo
# np.where sobre un array → devuelve array de strings de color
axes[1].bar(df_btc.index, df_btc['retorno'],
            color=np.where(df_btc['retorno'] >= 0, 'green', 'red'), alpha=0.5)
axes[1].set_title('Retorno diario (%)')
axes[1].axhline(0, color='black', linewidth=0.5)  # línea horizontal en 0

plt.tight_layout()
plt.show()

---
## Unidades 13-14 – Matplotlib: Principios y Tipos de Gráficos

### Principios de visualización (Tufte)
- **Claridad**: eliminar *chart junk* (grillas pesadas, 3D decorativo, sombras).
- **Integridad**: el eje Y de gráficos de barras **debe comenzar en 0** (de lo contrario exagera diferencias).
- **Contexto**: cada gráfico necesita título, etiquetas en los ejes y leyenda.
- **Jerarquía visual**: lo más importante tiene más tamaño, color o contraste.

### Anatomía de una figura Matplotlib
```
plt.figure()   ← contenedor principal
  └── Axes     ← 'el gráfico' con ejes, títulos, artistas
        └── Artists: Line2D, Rectangle (barra), Text, PathCollection (scatter)...
```

**API orientada a objetos** (recomendada): `fig, ax = plt.subplots()` → `ax.plot(...)`  
**API pyplot** (funcional, para scripts rápidos): `plt.plot(...)` directamente.

In [ ]:
# ── 1. Gráfico de líneas ─────────────────────────────────────────────────────
rng = np.random.default_rng(12)
meses = pd.date_range('2022-01', periods=24, freq='ME')

# cumsum de ruido gaussiano → caminata aleatoria (simula tendencia con variabilidad)
prod_A = 1000 + np.cumsum(rng.normal(50, 100, 24))
prod_B = 1200 + np.cumsum(rng.normal(30, 80,  24))

fig, ax = plt.subplots(figsize=(10, 4))

# marker='o': pone un círculo en cada dato → útil cuando hay pocos puntos
# ms=5: tamaño del marker en puntos tipográficos
ax.plot(meses, prod_A, marker='o', ms=5, label='Producto A')
ax.plot(meses, prod_B, marker='s', ms=5, label='Producto B')  # 's'=square

ax.set_title('Evolución de ventas mensuales')
ax.set_xlabel('Fecha')
ax.set_ylabel('Unidades')
ax.legend()
ax.tick_params(axis='x', rotation=45)   # rota etiquetas X para que no se superpongan

# FuncFormatter: da formato personalizado a las etiquetas del eje Y
# lambda x, _: el _ ignora el segundo argumento (posición del tick)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()   # ajusta márgenes automáticamente para que nada se corte
plt.show()

In [ ]:
# ── 2. Scatter con colorbar ──────────────────────────────────────────────────
n_sc = 200
x_sc = rng.normal(0, 1, n_sc)
y_sc = 2*x_sc + rng.normal(0, 0.5, n_sc)   # correlación lineal + ruido
color_sc = rng.uniform(0, 100, n_sc)        # tercera variable como color

fig, ax = plt.subplots(figsize=(7, 5))
# c=color_sc: asigna un color a cada punto según su valor
# cmap='viridis': mapa de color perceptualmente uniforme (accesible para daltónicos)
# edgecolors='none': sin borde en los puntos → más limpio cuando hay muchos
sc = ax.scatter(x_sc, y_sc, c=color_sc, cmap='viridis', alpha=0.7, edgecolors='none')

# colorbar: agrega la barra de referencia de color a la derecha
plt.colorbar(sc, ax=ax, label='Intensidad')
ax.set_title('Scatter con colorbar')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Barras agrupadas ──────────────────────────────────────────────────────
regiones = ['Norte', 'Sur', 'Centro', 'Este']
q1_vals  = rng.integers(400, 900, 4)
q2_vals  = rng.integers(400, 900, 4)

x_pos = np.arange(len(regiones))   # [0, 1, 2, 3] → posiciones base de cada grupo
w = 0.35                            # ancho de cada barra

fig, ax = plt.subplots(figsize=(8, 4))
# Para agrupar: desplazar Q1 a la izquierda (-w/2) y Q2 a la derecha (+w/2)
b1 = ax.bar(x_pos - w/2, q1_vals, width=w, label='Q1')
b2 = ax.bar(x_pos + w/2, q2_vals, width=w, label='Q2')

# set_xticks + set_xticklabels: reemplaza los números 0,1,2,3 por los nombres de región
ax.set_xticks(x_pos)
ax.set_xticklabels(regiones)
ax.set_title('Ventas por región y trimestre')
ax.set_ylabel('Ventas')
ax.legend()

# bar_label: anota el valor encima de cada barra (evita tener que leer el eje Y)
ax.bar_label(b1, fmt='%d', padding=2)
ax.bar_label(b2, fmt='%d', padding=2)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Histograma con curva KDE ─────────────────────────────────────────────
datos_hist = rng.normal(loc=50, scale=10, size=500)

fig, ax = plt.subplots(figsize=(8, 4))

# density=True: normaliza el histograma para que el área total = 1
# → permite superponer la curva de densidad (KDE) en la misma escala
# bins=30: número de intervalos (más bins = más detalle pero más ruido)
ax.hist(datos_hist, bins=30, density=True,
        color='steelblue', alpha=0.6, edgecolor='white', label='Histograma')

# KDE (Kernel Density Estimation): estimación no paramétrica de la distribución
# gaussian_kde usa un kernel gaussiano centrado en cada punto de dato
from scipy.stats import gaussian_kde
kde = gaussian_kde(datos_hist)   # ajusta el ancho de banda automáticamente
x_kde = np.linspace(datos_hist.min(), datos_hist.max(), 200)
ax.plot(x_kde, kde(x_kde), color='red', lw=2, label='KDE')

ax.set_title('Distribución con KDE')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Boxplot ─────────────────────────────────────────────────────────────
grupos_box = {
    'Grupo A': rng.normal(60, 10, 100),
    'Grupo B': rng.normal(75, 15, 100),  # mayor media y dispersión
    'Grupo C': rng.normal(50, 5,  100),  # menor dispersión
}

fig, ax = plt.subplots(figsize=(7, 4))
# patch_artist=True: rellena las cajas con color (por defecto son transparentes)
# boxprops/medianprops: personalización visual
ax.boxplot(
    grupos_box.values(),
    labels=grupos_box.keys(),
    patch_artist=True,
    boxprops=dict(facecolor='lightblue'),
    medianprops=dict(color='red', lw=2),   # mediana en rojo para mayor visibilidad
)
# Lectura del boxplot:
# Línea central = mediana | Caja = IQR (Q1 a Q3) | Bigotes = 1.5*IQR | Puntos = outliers
ax.set_title('Comparación de grupos (Boxplot)')
ax.set_ylabel('Valor')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Gráfico de torta (pie) ────────────────────────────────────────────────
labels_pie = ['Electrónica', 'Ropa', 'Alimentos', 'Hogar', 'Otros']
sizes_pie  = [35, 25, 20, 12, 8]   # deben sumar 100 (o se normalizan automáticamente)
explode    = [0.05] * 5            # separación de cada sector del centro

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    sizes_pie,
    labels=labels_pie,
    autopct='%1.1f%%',  # muestra el % con 1 decimal dentro de cada sector
    explode=explode,
    startangle=90,      # empieza desde las 12 en punto (más natural de leer)
    colors=sns.color_palette('pastel'),   # colores suaves de seaborn
)
# ⚠ Uso del pie: solo cuando hay pocas categorías (≤6) y se quiere mostrar proporción
# Para más categorías: usar barras horizontales ordenadas (más fácil de comparar)
ax.set_title('Participación por categoría')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Grid de subplots ──────────────────────────────────────────────────────
# plt.subplots(filas, columnas): crea una cuadrícula de Axes
# axes es un array numpy de forma (2,2) → se accede con axes[fila, col]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Subplot [0,0]: línea simple
axes[0, 0].plot(prod_A, color='steelblue')
axes[0, 0].set_title('Serie Producto A')

# Subplot [0,1]: barras verticales
axes[0, 1].bar(regiones, q1_vals, color='salmon')
axes[0, 1].set_title('Ventas Q1 por región')

# Subplot [1,0]: scatter simple (sin colorbar)
axes[1, 0].scatter(x_sc[:80], y_sc[:80], alpha=0.5, color='green')
axes[1, 0].set_title('Scatter simple')

# Subplot [1,1]: histograma
axes[1, 1].hist(datos_hist, bins=20, color='purple', alpha=0.7)
axes[1, 1].set_title('Histograma')

# axes.flat: itera sobre todos los Axes en orden → aplica grid a todos
for ax in axes.flat:
    ax.grid(True, alpha=0.3)   # grilla suave (alpha=0.3) para no dominar el gráfico

# suptitle: título global de la figura (por encima de todos los subplots)
plt.suptitle('Dashboard Matplotlib – 4 gráficos', fontsize=14, fontweight='bold')
plt.tight_layout()   # ajusta el espaciado entre subplots
plt.show()

---
## Unidad 15 – Plotly Express: Gráficos Interactivos

**Plotly** genera gráficos HTML con interactividad nativa:
- **Hover**: info al pasar el mouse
- **Zoom**: doble click para resetear
- **Export**: PNG desde el menú de la figura
- **write_html()**: guarda el gráfico como archivo `.html` autónomo (sin servidor)

| Función | Gráfico |
|---------|---------|
| `px.bar()` | Barras |
| `px.line()` | Líneas |
| `px.scatter()` | Dispersión |
| `px.pie(hole=0.4)` | Donut (agujero en el centro) |
| `px.treemap()` | Treemap jerárquico |
| `fig.write_html('file.html')` | Exportar a HTML |

In [ ]:
if PLOTLY:
    rng = np.random.default_rng(13)
    df_plotly = pd.DataFrame({
        'categoria': ['Electrónica', 'Ropa', 'Alimentos', 'Hogar', 'Libros',
                      'Deportes', 'Juguetes', 'Belleza'],
        'ventas_total': rng.integers(5000, 50000, 8),
        'crecimiento':  rng.uniform(-10, 30, 8).round(1),  # % de crecimiento
    })

    # ── Bar chart interactivo ────────────────────────────────────────────────
    # color='crecimiento': las barras se colorean según el crecimiento (escala RdYlGn)
    # → rojo = negativo, amarillo = neutral, verde = positivo
    # labels: diccionario para renombrar columnas en el hover y ejes
    fig_bar = px.bar(
        df_plotly.sort_values('ventas_total', ascending=False),  # de mayor a menor
        x='categoria',
        y='ventas_total',
        color='crecimiento',
        color_continuous_scale='RdYlGn',
        title='Ventas por categoría (interactivo)',
        labels={'ventas_total': 'Ventas', 'categoria': 'Categoría'},
    )
    fig_bar.show()   # muestra en el notebook; en script abre el navegador
else:
    print('Instalar plotly: pip install plotly')

In [ ]:
if PLOTLY:
    # ── Donut chart ──────────────────────────────────────────────────────────
    # hole=0.4: crea el agujero central (40% del radio) → convierte pie en donut
    # El donut es visualmente más moderno y el espacio central puede usarse para un KPI
    fig_pie = px.pie(
        df_plotly,
        names='categoria',
        values='ventas_total',
        hole=0.4,
        title='Participación de mercado (Donut)',
        color_discrete_sequence=px.colors.qualitative.Pastel,
    )
    # update_traces: modifica propiedades de todos los 'traces' (series de datos)
    # textinfo='percent+label': muestra % y nombre dentro de cada sector
    fig_pie.update_traces(textinfo='percent+label')
    fig_pie.show()

In [ ]:
if PLOTLY:
    # ── Treemap ───────────────────────────────────────────────────────────────
    # Treemap: visualiza jerarquías con rectángulos de área proporcional al valor
    # path: lista de columnas que forman la jerarquía (de mayor a menor nivel)
    # → sector > categoria (dos niveles de agrupamiento)
    df_plotly['sector'] = ['Tech', 'Moda', 'Comida', 'Hogar', 'Cultura',
                            'Deporte', 'Juego', 'Moda']
    fig_tree = px.treemap(
        df_plotly,
        path=['sector', 'categoria'],    # jerarquía: primero sector, luego categoría
        values='ventas_total',           # tamaño de cada rectángulo
        color='crecimiento',             # color según crecimiento
        color_continuous_scale='Blues',
        title='Treemap – Ventas por sector y categoría',
    )
    fig_tree.show()

In [ ]:
if PLOTLY:
    # ── Exportar a HTML ───────────────────────────────────────────────────────
    # write_html: guarda el gráfico completo (datos + interactividad) en un .html
    # → el archivo es autónomo: funciona sin Python ni servidor
    # → se puede abrir con cualquier navegador o incrustar en una página web
    fig_bar.write_html('ventas_plotly.html')
    print('Exportado: ventas_plotly.html')
    print('Abrir con el navegador para interacción completa.')

---
## Unidad 16 – Storytelling con Datos

### Estructura narrativa
1. **Problema / Pregunta de negocio** – ¿Qué queremos entender?
2. **Contexto** – Datos disponibles, período, alcance.
3. **Metodología** – ¿Cómo se analizó?
4. **Hallazgos clave** – 3-5 insights principales.
5. **Recomendaciones** – Acciones concretas derivadas.
6. **Limitaciones** – Sesgos, datos faltantes, supuestos.

### Adaptación al público
| Audiencia | Enfoque |
|-----------|---------|
| Ejecutivo | KPIs, impacto económico, próximas decisiones |
| Técnico | Metodología, métricas de modelo, reproducibilidad |
| Mixto | Resumen ejecutivo + apéndice técnico |

### Jerarquía visual en un dashboard
- El gráfico más importante va **arriba a la izquierda** (lectura natural en Z).
- Colores consistentes: el mismo color siempre representa lo mismo.
- Anotaciones directas en el gráfico > leyenda externa (menos movimiento ocular).

In [ ]:
# ── Datos del reporte ejecutivo ──────────────────────────────────────────────
rng = np.random.default_rng(14)
meses = pd.date_range('2023-01', periods=12, freq='ME')
categorias = ['Premium', 'Standard', 'Básico']

# Ventas con tendencia creciente (historia positiva para el ejecutivo)
ventas_total = np.array([120, 135, 128, 142, 138, 155, 160, 152, 170, 165, 178, 190]) * 1000

# Churn cayendo (otra historia positiva)
churn_rate = np.array([5.2, 4.8, 5.0, 4.5, 4.3, 4.0, 3.8, 4.1, 3.5, 3.2, 3.0, 2.8])

# Mix de clientes por categoría
mix_cat = pd.DataFrame({
    'Mes':      meses,
    'Premium':  rng.integers(30, 50, 12),
    'Standard': rng.integers(40, 60, 12),
    'Básico':   rng.integers(20, 40, 12),
}).set_index('Mes')

# NPS (Net Promoter Score): mide lealtad del cliente
# Fórmula: %Promotores - %Detractores. Rango: -100 a 100. Bueno si >50
nps = rng.integers(40, 80, 12)

# ── Dashboard 4 paneles ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Reporte Ejecutivo – 2023\nCrecimiento y Retención de Clientes',
             fontsize=15, fontweight='bold', y=1.01)

# ── Panel 1: Ventas (arriba-izquierda, el más importante) ─────────────────────
# fill_between: rellena el área bajo la curva → enfatiza el crecimiento acumulado
axes[0, 0].fill_between(meses, ventas_total/1e6, alpha=0.3, color='steelblue')
axes[0, 0].plot(meses, ventas_total/1e6, 'o-', color='steelblue')
axes[0, 0].set_title('Ventas mensuales (M$)')
axes[0, 0].set_ylabel('Millones ARS')
axes[0, 0].tick_params(axis='x', rotation=45)

# ── Panel 2: Churn rate (arriba-derecha) ─────────────────────────────────────
# Color semafórico: rojo = sobre el objetivo, verde = bajo el objetivo
color_churn = ['green' if c < 4.0 else 'red' for c in churn_rate]
axes[0, 1].bar(range(12), churn_rate, color=color_churn, alpha=0.7)
axes[0, 1].axhline(4.0, color='black', linestyle='--', lw=1, label='Objetivo 4%')
axes[0, 1].set_title('Tasa de Churn (%)')
axes[0, 1].set_ylabel('%')
axes[0, 1].set_xticks(range(12))
axes[0, 1].set_xticklabels([m.strftime('%b') for m in meses], rotation=45)
axes[0, 1].legend()

# ── Panel 3: Mix de clientes (abajo-izquierda) ───────────────────────────────
# Stacked bar: apila los segmentos → muestra composición total y mix simultáneamente
bottom = np.zeros(12)   # acumulador del 'piso' de cada barra
colors_stack = ['#2196F3', '#FF9800', '#4CAF50']   # azul, naranja, verde
for cat, col in zip(categorias, colors_stack):
    axes[1, 0].bar(range(12), mix_cat[cat], bottom=bottom, label=cat, color=col, alpha=0.85)
    bottom += mix_cat[cat].values   # próxima barra empieza donde termina la actual
axes[1, 0].set_title('Mix de clientes por categoría')
axes[1, 0].set_xticks(range(12))
axes[1, 0].set_xticklabels([m.strftime('%b') for m in meses], rotation=45)
axes[1, 0].legend(loc='upper left', fontsize=8)

# ── Panel 4: NPS (abajo-derecha) ─────────────────────────────────────────────
# fill_between con where: colorea distinto por encima y por debajo del umbral
axes[1, 1].plot(range(12), nps, 'D-', color='purple', ms=7)  # 'D'=diamond marker
axes[1, 1].fill_between(range(12), 50, nps,
                         where=(nps >= 50), alpha=0.2, color='green', label='NPS≥50')
axes[1, 1].fill_between(range(12), 50, nps,
                         where=(nps < 50), alpha=0.2, color='red',   label='NPS<50')
axes[1, 1].axhline(50, color='gray', lw=0.8, linestyle='--')   # umbral 'bueno'
axes[1, 1].set_title('Net Promoter Score (NPS)')
axes[1, 1].set_xticks(range(12))
axes[1, 1].set_xticklabels([m.strftime('%b') for m in meses], rotation=45)
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
# savefig: guarda en disco antes de show() (después de show el estado se resetea)
plt.savefig('reporte_ejecutivo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Reporte guardado como reporte_ejecutivo.png')

### Insights del Reporte

1. **Ventas**: crecimiento del 58% en 2023 (120k → 190k ARS mensuales).
2. **Churn**: cayó del 5.2% en enero al 2.8% en diciembre — la estrategia de retención funciona.
3. **Mix**: clientes Premium aumentaron proporcionalmente → mejora en valor de vida del cliente (LTV).
4. **NPS**: recuperación en H2 — correlaciona con la reducción de churn (clientes más satisfechos se van menos).

**Recomendación**: mantener la inversión en retención Premium; investigar los meses con churn > 4%.

---
## Desafío Final – HR Dataset

Aplicamos el flujo completo: **carga → EDA → visualización → pipeline de clasificación**.

In [ ]:
# ── Carga del dataset HR ──────────────────────────────────────────────────────
try:
    # El dataset real de Kaggle tiene columnas como:
    # satisfaction_level, last_evaluation, number_project, average_montly_hours,
    # time_spend_company, Work_accident, left, promotion_last_5years, Department, salary
    df_hr = pd.read_csv('HR_dataset.csv')
    print('HR Dataset cargado.')
except FileNotFoundError:
    # Generamos un dataset sintético con estructura similar
    rng = np.random.default_rng(15)
    n_hr = 800
    df_hr = pd.DataFrame({
        'employee_id': range(1, n_hr + 1),
        'departamento': rng.choice(['Ventas', 'IT', 'RRHH', 'Finanzas', 'Operaciones'], n_hr),
        'salario':      rng.integers(30000, 120000, n_hr),
        'antiguedad':   rng.integers(0, 20, n_hr),
        'satisfaccion': rng.choice([1, 2, 3, 4, 5], n_hr),  # escala Likert 1-5
        'evaluacion':   rng.uniform(0.4, 1.0, n_hr).round(2),  # 0=muy malo, 1=excelente
        # p=[0.85, 0.15]: solo el 15% fue promovido (dataset desbalanceado)
        'promovido':    rng.choice([0, 1], n_hr, p=[0.85, 0.15]),
    })
    print('HR Dataset sintético creado.')

print(df_hr.shape)   # (filas, columnas)
print(df_hr.dtypes)
print(df_hr.head())

In [ ]:
# ── EDA (Exploratory Data Analysis) ──────────────────────────────────────────
print('--- Estadísticas descriptivas ---')
# describe(): count, mean, std, min, 25%, 50%, 75%, max para numéricas
print(df_hr.describe().round(2))
print('\n--- Valores nulos ---')
# Siempre verificar nulos antes de modelar
print(df_hr.isnull().sum())

In [ ]:
# ── Dashboard HR: 4 visualizaciones ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Salario por departamento (boxplot)
# Permite comparar medianas y dispersiones entre departamentos
df_hr.boxplot(column='salario', by='departamento', ax=axes[0, 0])
axes[0, 0].set_title('Salario por departamento')
axes[0, 0].set_xlabel('')

# 2. Satisfacción vs Evaluación coloreado por promoción
# Hipótesis: empleados con alta satisfacción Y alta evaluación serán más promovidos
# cmap='coolwarm': 0=azul (no promovido), 1=rojo (promovido)
sc = axes[0, 1].scatter(
    df_hr['satisfaccion'], df_hr['evaluacion'],
    c=df_hr['promovido'], cmap='coolwarm', alpha=0.5, edgecolors='none'
)
axes[0, 1].set_title('Satisfacción vs Evaluación (rojo=promovido)')
axes[0, 1].set_xlabel('Satisfacción (1-5)')
axes[0, 1].set_ylabel('Evaluación (0-1)')
plt.colorbar(sc, ax=axes[0, 1])

# 3. Distribución de antigüedad (histograma)
# ¿Es uniforme o hay un pico en ciertos años? → puede indicar rotación
axes[1, 0].hist(df_hr['antiguedad'], bins=20, color='teal', edgecolor='white', alpha=0.8)
axes[1, 0].set_title('Distribución de antigüedad')
axes[1, 0].set_xlabel('Años')

# 4. Tasa de promoción por departamento
# .mean() sobre columna binaria (0/1) = proporción de 1s = tasa de promoción
promo_rate = df_hr.groupby('departamento')['promovido'].mean() * 100
promo_rate.sort_values(ascending=False).plot(
    kind='bar', ax=axes[1, 1], color='steelblue', edgecolor='white'
)
axes[1, 1].set_title('Tasa de promoción por departamento (%)')
axes[1, 1].set_ylabel('%')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('Análisis HR Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Pipeline de clasificación: predecir si un empleado será promovido ─────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

features_hr = ['salario', 'antiguedad', 'satisfaccion', 'evaluacion']
X_hr = df_hr[features_hr]
y_hr = df_hr['promovido']   # variable target: 0=no promovido, 1=promovido

X_tr, X_te, y_tr, y_te = train_test_split(X_hr, y_hr, test_size=0.2, random_state=42)

pipe_hr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # por si hay NaN en los features
    ('scaler',  StandardScaler()),                   # necesario para Logistic Regression
    ('clf',     LogisticRegression(
        max_iter=1000,          # aumentar si no converge con el default (100)
        class_weight='balanced' # pondera más la clase minoritaria (promovidos=15%)
        # sin 'balanced', el modelo podría predecir siempre 0 y tener 85% accuracy
    )),
])

pipe_hr.fit(X_tr, y_tr)
y_pred = pipe_hr.predict(X_te)

# classification_report: precision, recall y F1 por clase
# Precision: de los que predije como promovidos, ¿cuántos realmente lo fueron?
# Recall: de los que realmente fueron promovidos, ¿cuántos detecté?
# F1: media armónica de precision y recall → balance entre ambas
print(classification_report(y_te, y_pred, target_names=['No promovido', 'Promovido']))